In [2]:
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [3]:
clientes = catalog.load("clientes")
productos = catalog.load("productos")
ventas = catalog.load("ventas")
devoluciones = catalog.load("devoluciones")

[04/07/26 15:25:40] INFO     Loading data from clientes (CSVDataset)...                        data_catalog.py:1048

                    INFO     Loading data from productos (CSVDataset)...                       data_catalog.py:1048

                    INFO     Loading data from ventas (CSVDataset)...                          data_catalog.py:1048

                    INFO     Loading data from devoluciones (CSVDataset)...                    data_catalog.py:1048

In [5]:
clientes

,id_cliente,nombre,email,region,ciudad,fecha_registro,segmento
0,NaN,CristÃ³bal Silva MuÃ±oz,cristÃ³bal.silva@gmail.com,Los Lagos,Osorno,2020-01-01,Regular
1,2.0,Fernanda GarcÃ­a HernÃ¡ndez,fernanda.garcÃ­a@outlook.com,Coquimbo,La Serena,2020-01-03,Regular
2,3.0,AndrÃ©s Contreras Soto,NaN,O'Higgins,Rengo,2020-01-05,Regular
3,NaN,Natalia Tapia Rojas,natalia.tapia@gmail.com,Los Lagos,Osorno,07/01/2020,VIP
4,5.0,JosÃ© Soto Fuentes,josÃ©.soto@gmail.com,BiobÃ­o,ConcepciÃ³n,2020-01-09,Premium
...,...,...,...,...,...,...,...
304,276.0,Marcela Bravo Rojas,marcela.bravo@gmail.com,Metropolitana,Santiago,2021-07-04,Nuevo
305,2.0,Fernanda GarcÃ­a HernÃ¡ndez,fernanda.garcÃ­a@outlook.com,Coquimbo,La Serena,2020-01-03,Regular
306,88.0,Ana Flores Soto,ana.flores@gmail.com,Los Lagos,Puerto Montt,23/06/2020,Regular
307,287.0,Daniela Flores GonzÃ¡lez,daniela.flores@gmail.com,O'Higgins,Rengo,2021-07-26,Premium


In [6]:
import pandas as pd
import numpy as np

def limpiar_texto(df):
    for col in df.select_dtypes(include="object").columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[\x00-\x1F\x7F-\x9F]", "", regex=True)
            .str.strip()
        )
    return df

In [7]:
clientes_clean = limpiar_texto(clientes)
productos_clean = limpiar_texto(productos)
ventas_clean = limpiar_texto(ventas)
devoluciones_clean = limpiar_texto(devoluciones)

In [8]:
clientes_clean = clientes_clean.drop_duplicates()
productos_clean = productos_clean.drop_duplicates()
ventas_clean = ventas_clean.drop_duplicates()
devoluciones_clean = devoluciones_clean.drop_duplicates()

In [10]:
# CLIENTES
clientes_clean.loc[:, "id_cliente"] = pd.to_numeric(
    clientes_clean["id_cliente"], errors="coerce"
)

clientes_clean.loc[:, "fecha_registro"] = pd.to_datetime(
    clientes_clean["fecha_registro"], errors="coerce"
)

In [13]:
clientes_clean.isnull().sum()


id_cliente        0
nombre            0
email             0
region            0
ciudad            0
fecha_registro    0
segmento          0
dtype: int64

In [12]:
clientes_clean = clientes_clean.dropna()

In [14]:
clientes_clean.head()

,id_cliente,nombre,email,region,ciudad,fecha_registro,segmento
1,2.0,Fernanda GarcÃ­a HernÃ¡ndez,fernanda.garcÃ­a@outlook.com,Coquimbo,La Serena,2020-01-03,Regular
2,3.0,AndrÃ©s Contreras Soto,nan,O'Higgins,Rengo,2020-01-05,Regular
4,5.0,JosÃ© Soto Fuentes,josÃ©.soto@gmail.com,BiobÃ­o,ConcepciÃ³n,2020-01-09,Premium
7,8.0,SebastiÃ¡n Valenzuela Bravo,sebastiÃ¡n.valenzuela@gmail.com,Metropolitana,Santiago,2020-01-15,Regular
11,12.0,Valentina Castro Contreras,valentina.castro@gmail.com,Maule,CuricÃ³,2020-01-23,premium


In [17]:
ventas_clean.loc[:, "cantidad"] = pd.to_numeric(
    ventas_clean["cantidad"], errors="coerce"
)

ventas_clean.loc[:, "precio_unitario"] = pd.to_numeric(
    ventas_clean["precio_unitario"], errors="coerce"
)

ventas_clean.loc[:, "fecha"] = pd.to_datetime(
    ventas_clean["fecha"], errors="coerce"
)

ventas_clean.loc[:, "total_venta"] = (
    ventas_clean["cantidad"] * ventas_clean["precio_unitario"]
)

In [18]:
ventas_clean.head()

,id_venta,fecha,id_cliente,id_producto,cantidad,precio_unitario,metodo_pago,canal_venta,total_venta
0,NaN,2023-01-01,26.0,128.0,1.0,411094.0,Tarjeta CrÃ©dito,Web,411094.0
1,2.0,NaT,47.0,111.0,8.0,330096.0,Tarjeta DÃ©bito,Web,2640768.0
2,3.0,2023-02-01,288.0,136.0,19.0,398324.0,WEBPAY,Tienda FÃ­sica,7568156.0
3,4.0,NaT,266.0,106.0,NaN,136701.0,Webpay,Tienda FÃ­sica,NaN
4,5.0,2023-03-01,16.0,186.0,3.0,346487.0,Webpay,Marketplace,1039461.0


In [19]:
df_final = ventas_clean.merge(
    clientes_clean,
    on="id_cliente",
    how="left"
)

df_final = df_final.merge(
    productos_clean,
    on="id_producto",
    how="left"
)

In [20]:
df_final.head()

,id_venta,fecha,id_cliente,id_producto,cantidad,precio_unitario,metodo_pago,canal_venta,total_venta,nombre_x,...,region,ciudad,fecha_registro,segmento,nombre_y,categoria,subcategoria,precio_lista,stock,proveedor
0,NaN,2023-01-01,26.0,128.0,1.0,411094.0,Tarjeta CrÃ©dito,Web,411094.0,Ana HernÃ¡ndez Rojas,...,O'Higgins,Rancagua,2020-02-20,Nuevo,Producto_128,Hogar,Sub_A,160705.0,448.0,ProvE
1,2.0,NaT,47.0,111.0,8.0,330096.0,Tarjeta DÃ©bito,Web,2640768.0,NaN,...,NaN,NaN,NaT,NaN,Producto_111,Ropa,Sub_A,89542.0,484.0,ProvC
2,3.0,2023-02-01,288.0,136.0,19.0,398324.0,WEBPAY,Tienda FÃ­sica,7568156.0,NaN,...,NaN,NaN,NaT,NaN,Producto_136,Ropa,Sub_C,475353.0,38.0,ProvF
3,4.0,NaT,266.0,106.0,NaN,136701.0,Webpay,Tienda FÃ­sica,NaN,Camila HernÃ¡ndez SepÃºlveda,...,Metropolitana,MaipÃº,2021-06-14,Nuevo,Producto_106,Alimentos,Sub_C,314842.0,132.0,ProvD
4,5.0,2023-03-01,16.0,186.0,3.0,346487.0,Webpay,Marketplace,1039461.0,NaN,...,NaN,NaN,NaT,NaN,Producto_186,Belleza,Sub_A,373276.0,334.0,ProvB


In [21]:
def corregir_encoding(df):
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(
            lambda x: x.encode("latin1").decode("utf-8") if isinstance(x, str) else x
        )
    return df

In [23]:
def corregir_texto_seguro(valor):
    """
    Intenta corregir textos mal decodificados.
    Si no se puede corregir, devuelve el valor original.
    """
    if not isinstance(valor, str):
        return valor

    try:
        return valor.encode("latin1").decode("utf-8")
    except:
        return valor


def corregir_encoding(df):
    """
    Aplica corrección de texto solo en columnas tipo object.
    """
    df = df.copy()

    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(corregir_texto_seguro)

    return df

In [24]:
clientes_clean = corregir_encoding(clientes_clean)
productos_clean = corregir_encoding(productos_clean)
ventas_clean = corregir_encoding(ventas_clean)
devoluciones_clean = corregir_encoding(devoluciones_clean)

In [25]:
clientes_clean.head()

,id_cliente,nombre,email,region,ciudad,fecha_registro,segmento
1,2.0,Fernanda García Hernández,fernanda.garcía@outlook.com,Coquimbo,La Serena,2020-01-03,Regular
2,3.0,Andrés Contreras Soto,nan,O'Higgins,Rengo,2020-01-05,Regular
4,5.0,José Soto Fuentes,josé.soto@gmail.com,Biobío,Concepción,2020-01-09,Premium
7,8.0,Sebastián Valenzuela Bravo,sebastián.valenzuela@gmail.com,Metropolitana,Santiago,2020-01-15,Regular
11,12.0,Valentina Castro Contreras,valentina.castro@gmail.com,Maule,Curicó,2020-01-23,premium
